In [1]:
import pandas as pd
import numpy as np
import random
from datetime import datetime, timedelta

# Set seed for reproducibility
np.random.seed(42)
random.seed(42)

# Configuration Variables
NUM_RECORDS = 1000
START_DATE = datetime(2026, 1, 1)

# Real-world Healthcare Domain Mappings
CLINICS = ['Cardiology', 'Pediatrics', 'General Medicine', 'Orthopedics', 'Dermatology']
INSURANCE_PROVIDERS = ['Daman', 'AXA Gulf', 'Oman Insurance', 'Bupa Global', 'NextCare']

# Valid ICD-10 (Diagnosis) to CPT (Procedure) mapping with SNOMED descriptions
CLINICAL_PROFILES = [
    {"icd10": "I10", "snomed": "Essential hypertension", "cpt": "99213", "base_cost": 250},      # Outpatient visit
    {"icd10": "E11.9", "snomed": "Type 2 diabetes mellitus", "cpt": "83036", "base_cost": 150},  # HbA1c Test
    {"icd10": "J06.9", "snomed": "Acute upper respiratory infection", "cpt": "99212", "base_cost": 180},
    {"icd10": "M17.11", "snomed": "Unilateral osteoarthritis, right knee", "cpt": "73562", "base_cost": 450}, # X-ray knee
    {"icd10": "L21.9", "snomed": "Seborrheic dermatitis", "cpt": "11102", "base_cost": 600}      # Skin biopsy
]

DENIAL_REASONS = ['Missing Prior Authorization', 'Incorrect Modifier', 'Patient Not Covered', 'Duplicate Claim']

print("Generating synthetic healthcare datasets for HealthHub...")

# --- TABLE 1: ENCOUNTERS (Clinical & Operations) ---
encounters_data = []

for i in range(NUM_RECORDS):
    encounter_id = f"ENC-{100000 + i}"
    patient_id = f"PAT-{random.randint(5000, 9999)}"
    
    # Operational metrics
    clinic = random.choice(CLINICS)
    days_offset = random.randint(0, 260) # Spread across 2026
    appointment_date = START_DATE + timedelta(days=days_offset, hours=random.randint(8, 17))
    wait_time_minutes = int(np.random.exponential(scale=15) + 5) # Skewed wait times
    
    # High risk of no-show if appointment lag is long (simulated behavior)
    no_show_prob = 0.15 if wait_time_minutes < 30 else 0.45
    no_show = 1 if random.random() < no_show_prob else 0
    
    # Pick a random clinical profile
    profile = random.choice(CLINICAL_PROFILES)
    
    encounters_data.append({
        "Encounter_ID": encounter_id,
        "Patient_ID": patient_id,
        "Appointment_DateTime": appointment_date.strftime('%Y-%m-%d %H:%M'),
        "Clinic_Specialty": clinic,
        "Waiting_Room_Minutes": wait_time_minutes if no_show == 0 else 0,
        "ICD10_Code": profile["icd10"],
        "SNOMED_Description": profile["snomed"],
        "No_Show_Flag": no_show
    })

df_encounters = pd.DataFrame(encounters_data)

# --- TABLE 2: BILLING & REVENUE CYCLE (RCM) ---
billing_data = []

# Only generate billing rows for patients who actually showed up
df_attended = df_encounters[df_encounters["No_Show_Flag"] == 0]

for idx, row in df_attended.iterrows():
    tx_id = f"TX-{200000 + idx}"
    
    # Match the clinical profile back to calculate cost
    profile = next(p for p in CLINICAL_PROFILES if p["icd10"] == row["ICD10_Code"])
    gross_amount = profile["base_cost"] + random.randint(-20, 100) # Minor cost variation
    
    # Simulation logic for claims denials (RCM data science feature)
    insurance = random.choice(INSURANCE_PROVIDERS)
    
    # Simulate a high denial rate on specific costly procedures to mimic real data irregularities
    is_denied = random.random() < 0.12 if gross_amount < 400 else random.random() < 0.35
    claim_status = "Rejected" if is_denied else "Approved"
    denial_reason = random.choice(DENIAL_REASONS) if claim_status == "Rejected" else "N/A"
    
    billing_data.append({
        "Transaction_ID": tx_id,
        "Encounter_ID": row["Encounter_ID"],
        "CPT_Code": profile["cpt"],
        "Insurance_Carrier": insurance,
        "Gross_Amount_AED": round(gross_amount, 2),
        "Claim_Status": claim_status,
        "Denial_Reason": denial_reason
    })

df_billing = pd.DataFrame(billing_data)

# --- SAVE DATASETS ---
df_encounters.to_csv("fact_encounters.csv", index=False)
df_billing.to_csv("fact_billing.csv", index=False)

print(print("Done! 'fact_encounters.csv' and 'fact_billing.csv' generated successfully."))
print(f"Summary: Generated {len(df_encounters)} encounters and {len(df_billing)} processed claims.")


Generating synthetic healthcare datasets for HealthHub...
Done! 'fact_encounters.csv' and 'fact_billing.csv' generated successfully.
None
Summary: Generated 1000 encounters and 804 processed claims.
